In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/otto-recommender-system/sample_submission.csv
/kaggle/input/competitions/otto-recommender-system/test.jsonl
/kaggle/input/competitions/otto-recommender-system/train.jsonl
/kaggle/input/datasets/radek1/otto-full-optimized-memory-footprint/id2type.pkl
/kaggle/input/datasets/radek1/otto-full-optimized-memory-footprint/test.parquet
/kaggle/input/datasets/radek1/otto-full-optimized-memory-footprint/train.parquet
/kaggle/input/datasets/radek1/otto-full-optimized-memory-footprint/type2id.pkl


In [2]:
import pandas as pd, numpy as np, gc, pickle
from collections import defaultdict

PATH = '/kaggle/input/datasets/radek1/otto-full-optimized-memory-footprint/'
train = pd.read_parquet(PATH + 'train.parquet')
test  = pd.read_parquet(PATH + 'test.parquet')

with open(PATH + 'type2id.pkl', 'rb') as f:
    type2id = pickle.load(f)
print(type2id)

print(train.shape, test.shape)
print(train.head())
print(train['type'].value_counts())

{'clicks': 0, 'carts': 1, 'orders': 2}
(216716096, 4) (6928123, 4)
   session      aid          ts  type
0        0  1517085  1659304800     0
1        0  1563459  1659304904     0
2        0  1309446  1659367439     0
3        0    16246  1659367719     0
4        0  1781822  1659367871     0
type
0    194720954
1     16896191
2      5098951
Name: count, dtype: int64


In [3]:
import time
t0 = time.time()

def build_covisitation(df, topk=20, last_n=20, max_diff=24*3600, n_chunks=10):
    # garder seulement les last_n derniers événements de chaque session
    df = df.sort_values(['session', 'ts'], ascending=[True, False]).copy()
    df['rank'] = df.groupby('session').cumcount()
    df = df[df['rank'] < last_n][['session', 'aid', 'ts']]

    sessions = df['session'].unique()
    chunks = np.array_split(sessions, n_chunks)
    pair_counts = None

    for i, ch in enumerate(chunks):
        sub = df[df['session'].isin(ch)]
        m = sub.merge(sub, on='session')
        m = m[(m['aid_x'] != m['aid_y']) &
              ((m['ts_x'] - m['ts_y']).abs() < max_diff)]
        c = m.groupby(['aid_x', 'aid_y']).size().rename('w')
        pair_counts = c if pair_counts is None else pair_counts.add(c, fill_value=0)
        del sub, m, c; gc.collect()
        print(f'chunk {i+1}/{n_chunks} fini ({time.time()-t0:.0f}s)')

    pair_counts = pair_counts.reset_index()
    pair_counts = pair_counts.sort_values(['aid_x', 'w'], ascending=[True, False])
    pair_counts['r'] = pair_counts.groupby('aid_x').cumcount()
    pair_counts = pair_counts[pair_counts['r'] < topk]

    neighbors = defaultdict(list)
    for a, b, w in zip(pair_counts.aid_x, pair_counts.aid_y, pair_counts.w):
        neighbors[a].append((b, w))
    return neighbors

# Échantillon de 300 000 sessions (RAM)
all_sessions = train['session'].unique()
rng = np.random.default_rng(42)
sample_sessions = rng.choice(all_sessions, size=300_000, replace=False)

sample_df = train[train['session'].isin(sample_sessions)]
print('échantillon:', sample_df.shape)

neighbors = build_covisitation(sample_df)
print('produits avec voisins:', len(neighbors))
print(f'temps total: {time.time()-t0:.0f}s')

del sample_df; gc.collect()

échantillon: (5066006, 4)
chunk 1/10 fini (8s)
chunk 2/10 fini (10s)
chunk 3/10 fini (13s)
chunk 4/10 fini (17s)
chunk 5/10 fini (22s)
chunk 6/10 fini (27s)
chunk 7/10 fini (33s)
chunk 8/10 fini (40s)
chunk 9/10 fini (47s)
chunk 10/10 fini (55s)
produits avec voisins: 491747
temps total: 62s


0

In [4]:
t0 = time.time()

# Top 20 produits les plus cliqués (fallback)
top_clicks = train[train['type'] == 0]['aid'].value_counts().index[:20].tolist()

def predict_session(aids):
    seen = list(dict.fromkeys(aids[::-1]))        # uniques, du plus récent au plus ancien
    scores = defaultdict(float)
    for rank, a in enumerate(seen[:10]):
        for b, w in neighbors.get(a, []):
            scores[b] += w / (rank + 1)           # les événements récents pèsent plus
    ranked = [a for a, _ in sorted(scores.items(), key=lambda x: -x[1])]
    result = seen[:20]
    for a in ranked + top_clicks:
        if len(result) >= 20: break
        if a not in result: result.append(a)
    return result[:20]

test_sorted = test.sort_values(['session', 'ts'])
sess_aids = test_sorted.groupby('session')['aid'].apply(list)
print('sessions test:', len(sess_aids), f'({time.time()-t0:.0f}s)')

preds = {s: predict_session(a) for s, a in sess_aids.items()}
print(f'prédictions finies ({time.time()-t0:.0f}s)')

rows = []
for s, labels in preds.items():
    lab = ' '.join(map(str, labels))
    for t in ['clicks', 'carts', 'orders']:
        rows.append((f'{s}_{t}', lab))

sub = pd.DataFrame(rows, columns=['session_type', 'labels'])
sub.to_csv('submission.csv', index=False)
print(sub.shape)
print(sub.head())

sessions test: 1671803 (49s)
prédictions finies (106s)
(5015409, 2)
      session_type                                             labels
0  12899779_clicks  59625 1460571 108125 29735 485256 1733943 1849...
1   12899779_carts  59625 1460571 108125 29735 485256 1733943 1849...
2  12899779_orders  59625 1460571 108125 29735 485256 1733943 1849...
3  12899780_clicks  1142000 736515 973453 582732 1502122 1263108 8...
4   12899780_carts  1142000 736515 973453 582732 1502122 1263108 8...


In [5]:
t0 = time.time()

# 1) Plus de sessions + sessions du test
rng = np.random.default_rng(42)
sample_sessions = rng.choice(train['session'].unique(), size=1_000_000, replace=False)
big_df = pd.concat([
    train[train['session'].isin(sample_sessions)][['session', 'aid', 'ts']],
    test[['session', 'aid', 'ts']]
], ignore_index=True)
print('data co-visitation:', big_df.shape, f'({time.time()-t0:.0f}s)')

neighbors = build_covisitation(big_df, topk=20, last_n=20, n_chunks=20)
print('produits avec voisins:', len(neighbors), f'({time.time()-t0:.0f}s)')
del big_df; gc.collect()

# 2) Prédiction avec poids par type + normalisation
TYPE_W = {0: 1.0, 1: 6.0, 2: 3.0}   # clicks, carts, orders

def predict_session_v2(aids, types):
    aids, types = aids[::-1][:20], types[::-1][:20]
    scores = defaultdict(float)
    for i, (a, t) in enumerate(zip(aids, types)):
        wt = TYPE_W[t] / (i + 1)
        scores[a] += wt
        nb = neighbors.get(a)
        if nb:
            tot = sum(w for _, w in nb)
            for b, w in nb:
                scores[b] += wt * w / tot
    ranked = [a for a, _ in sorted(scores.items(), key=lambda x: -x[1])]
    result = ranked[:20]
    for a in top_clicks:
        if len(result) >= 20: break
        if a not in result: result.append(a)
    return result[:20]

test_sorted = test.sort_values(['session', 'ts'])
sess = test_sorted.groupby('session').agg({'aid': list, 'type': list})
print('sessions test:', len(sess), f'({time.time()-t0:.0f}s)')

sessions = sess.index.values
labels_all = [' '.join(map(str, predict_session_v2(a, t)))
              for a, t in zip(sess['aid'].values, sess['type'].values)]
print(f'prédictions finies ({time.time()-t0:.0f}s)')

sub = pd.DataFrame({
    'session_type': np.repeat(sessions, 3).astype(str),
    'labels': np.repeat(labels_all, 3)
})
sub['session_type'] = sub['session_type'] + np.tile(['_clicks', '_carts', '_orders'], len(sessions))
sub.to_csv('submission.csv', index=False)
print(sub.shape)
print(sub.head())

data co-visitation: (23811866, 3) (5s)
chunk 1/20 fini (24s)
chunk 2/20 fini (41s)
chunk 3/20 fini (60s)
chunk 4/20 fini (83s)
chunk 5/20 fini (109s)
chunk 6/20 fini (137s)
chunk 7/20 fini (167s)
chunk 8/20 fini (201s)
chunk 9/20 fini (234s)
chunk 10/20 fini (267s)
chunk 11/20 fini (302s)
chunk 12/20 fini (337s)
chunk 13/20 fini (372s)
chunk 14/20 fini (409s)
chunk 15/20 fini (447s)
chunk 16/20 fini (490s)
chunk 17/20 fini (532s)
chunk 18/20 fini (575s)
chunk 19/20 fini (620s)
chunk 20/20 fini (665s)
produits avec voisins: 1113568 (686s)
sessions test: 1671803 (741s)
prédictions finies (813s)
(5015409, 2)
      session_type                                             labels
0  12899779_clicks  59625 1460571 108125 29735 485256 1733943 1849...
1   12899779_carts  59625 1460571 108125 29735 485256 1733943 1849...
2  12899779_orders  59625 1460571 108125 29735 485256 1733943 1849...
3  12899780_clicks  1142000 736515 582732 973453 487136 1758603 88...
4   12899780_carts  1142000 736515 58